In [1]:
SMOKE = False
JOBS = ['12-24', '25-37']
import glob, os, subprocess
t = glob.glob('/kaggle/input/**/leakbench/train.py', recursive=True)[0]
ROOT = t.rsplit('/leakbench/', 1)[0]
print('ROOT', ROOT)
subprocess.run('pip -q install segmentation-models-pytorch==0.5.0 timm', shell=True)
subprocess.run(f'rm -rf /kaggle/working/leakbench && cp -r {ROOT}/leakbench /kaggle/working/leakbench', shell=True)
W = '/kaggle/working/leakbench'
if SMOKE: cmds = [f'CUDA_VISIBLE_DEVICES={g} python {W}/train.py --data-root {ROOT} --manifest {W}/manifests/v2.csv --model {m} --epochs 1 --limit 32 --out /kaggle/working/smoke' for g, m in ((0, 'unet_r34'), (1, 'xattn'))]
else: cmds = [f'CUDA_VISIBLE_DEVICES={g} python {W}/run_queue.py --data-root {ROOT} --jobs {j} --out /kaggle/working/runs --budget-hours 11.5 > /kaggle/working/log_gpu{g}.txt 2>&1' for g, j in enumerate(JOBS)]
procs = [subprocess.Popen(c, shell=True) for c in cmds]
print([p.wait() for p in procs])
subprocess.run(f'python {W}/aggregate.py ' + ('/kaggle/working/smoke' if SMOKE else '/kaggle/working/runs'), shell=True)

ROOT /kaggle/input/datasets/gokulrocky/chakra-leakbench-v1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.1 MB/s eta 0:00:00
[0, 0]
E3 segformer_b2: random-split minus video-grouped ClinicDB Dice = +0.051 (folds=5, per-fold [0.011, -0.006, -0.017, 0.1, 0.168])
E3 unet_r34: random-split minus video-grouped ClinicDB Dice = +0.068 (folds=5, per-fold [0.009, 0.16, 0.121, 0.037, 0.011])


CompletedProcess(args='python /kaggle/working/leakbench/aggregate.py /kaggle/working/runs', returncode=0)